# 32 – LODES Jobs + Geometry Join

This notebook combines:

- **LODES spatial units** (blocks / block groups),
- **Jobs and earnings data** from LODES WAC.

**Goals:**
- Load cleaned LODES spatial units and jobs table.
- Join jobs onto geometries.
- Optionally compute basic densities (jobs per acre).
- Save a jobs layer ready for intersecting to tracts and then overlaying to TOCs/villages/etc.


In [7]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

JOBS_DIR = DATA_DIR / "jobs" / "lodes_processed"

lodes_geom_clean_path = JOBS_DIR / "lodes_units_clean.gpkg"
jobs_table_path = JOBS_DIR / "lodes_jobs_maricopa_2022.csv"

lodes_geom_clean_path, jobs_table_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/lodes_units_clean.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/lodes_jobs_maricopa_2022.csv'))

In [8]:
lodes_units = gpd.read_file(lodes_geom_clean_path)
jobs_blocks = pd.read_csv(jobs_table_path)

print("LODES spatial units:", len(lodes_units))
print("Jobs records:", len(jobs_blocks))

LODES spatial units: 2806
Jobs records: 22090


In [9]:
lodes_units.columns

Index(['GEOID', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE', 'geometry'], dtype='object')

In [10]:
jobs_blocks.columns

Index(['w_geocode', 'C000', 'CE01', 'CE02', 'CE03'], dtype='object')

In [ ]:
# making sure w_geocode is string and zero-padded to 15 characters

jobs_blocks["w_geocode"] = jobs_blocks["w_geocode"].astype(str).str.zfill(15)
jobs_blocks["BG_GEOID"] = jobs_blocks["w_geocode"].str.slice(0, 12)

In [24]:
# the columns I need:

job_cols = [
    "C000",   # total jobs
    "CE01",   # earnings bin 1
    "CE02",   # earnings bin 2
    "CE03",   # earnings bin 3
]

In [39]:
jobs_block_groups = (
    jobs_blocks
    .groupby("BG_GEOID", as_index=False)[job_cols]
    .sum()
)

jobs_block_groups.head()

,BG_GEOID,C000,CE01,CE02,CE03
0,040130101021,455,66,114,275
1,040130101022,233,26,54,153
2,040130101023,1150,144,429,577
3,040130101031,121,21,23,77
4,040130101032,235,41,78,116


In [40]:
lodes_units["GEOID"] = lodes_units["GEOID"].astype(str)

lodes_with_jobs = lodes_units.merge(
    jobs_block_groups,
    left_on="GEOID",
    right_on="BG_GEOID",
    how="left"
).drop(columns=["BG_GEOID"])

lodes_with_jobs.head()

,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,geometry,C000,CE01,CE02,CE03
0,040130101021,04,013,010102,1,"POLYGON ((827801.708 1091626.91, 827865.552 10...",455,66,114,275
1,040130101021,04,013,010102,2,"POLYGON ((725542.865 1023666.739, 723744.814 1...",455,66,114,275
2,040130101021,04,013,010102,3,"POLYGON ((712735.735 1097380.426, 712738.472 1...",455,66,114,275
3,040130101021,04,013,010103,1,"POLYGON ((744671.443 1002586.263, 744681.625 1...",455,66,114,275
4,040130101021,04,013,010103,2,"POLYGON ((765755.318 1002785.693, 765592.367 1...",455,66,114,275


In [41]:
# Ensure CRS is projected (meters/feet); if not, stop and fix CRS first.
print("CRS:", lodes_with_jobs.crs)

CRS: EPSG:2868


In [ ]:
# Since CRS is the project CRS, time to compute area.

lodes_with_jobs["area_sq_ft"] = lodes_with_jobs.geometry.area

# Convert sq feet to acres (1 acre ≈ 43560 sq_ft)
lodes_with_jobs["area_acres"] = lodes_with_jobs["area_sq_ft"] / 43560

# Jobs per acre (watch out for division by zero)
lodes_with_jobs["jobs_per_acre"] = lodes_with_jobs["C000"] / lodes_with_jobs["area_acres"]

lodes_with_jobs[["C000", "area_acres", "jobs_per_acre"]].head()

,C000,area_acres,jobs_per_acre
0,455,593324.922484,0.000767
1,455,2929.334492,0.155325
2,455,80707.955066,0.005638
3,455,3795.674584,0.119873
4,455,5508.309491,0.082602


In [46]:
# double checking to make sure the rows and unique GEOIDs match, I was a bit confused by the above C000 return

print("Jobs BG rows:", len(jobs_block_groups))
print("Unique BG_GEOID:", jobs_block_groups["BG_GEOID"].nunique())

Jobs BG rows: 2782
Unique BG_GEOID: 2782


In [47]:
lodes_jobs_path = JOBS_DIR / "lodes_jobs_2022_with_geometry.gpkg"
lodes_with_jobs.to_file(lodes_jobs_path, driver="GPKG")

print("Saved LODES jobs + geometry to:", lodes_jobs_path)

Saved LODES jobs + geometry to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\jobs\lodes_processed\lodes_jobs_2022_with_geometry.gpkg
